# M2CAN de bout en bout sur l'esthétique (Colab)

M2CAN **dans son propre protocole** : BERT et ResNet-50 **entraînés**, configuration publiée de chaque cible
(`msmm/scripts/config_<cible>.yaml`), lots de 8, 10 époques, graine publiée 123456. Le code est celui du dépôt
MM-WBHR (`m2can/`, code des auteurs). But : mesurer M2CAN sur nos découpes, à comparer à l'article
(79,9 / 69,8 / 74,5) et à nos résultats sur caractéristiques gelées.

**Avant de lancer :** *Exécution > Modifier le type d'exécution > GPU* (A100 ou L4 de préférence). Puis
*Exécution > Tout exécuter*.

**Données :** Colab télécharge lui-même les images (Google Drive des auteurs, environ 5,6 Go) et les découpes
(GitHub des auteurs, empreintes vérifiées). Rien ne passe par votre ordinateur.

**Sorties :** sur votre Google Drive, dans `M2CAN_bout_en_bout/` : journaux, `resultats.json` par cible et un
récapitulatif. Si la session coupe, relancer *Tout exécuter* : les cibles déjà terminées ne sont pas refaites.

## Paramètres

In [ ]:
import os
DEPOT        = "https://github.com/Kevindjoum/MM-WBHR.git"
BRANCHE      = "main"
SORTIE       = "/content/drive/MyDrive/M2CAN_bout_en_bout"   # sur votre Google Drive
CIBLES       = ["AVA", "PCCD", "RPCD"]                        # l'ordre = la priorité
GRAINE       = 123456                                         # graine publiée de M2CAN
EPOQUES      = 10                                             # valeur publiée
LOT          = 8                                              # valeur publiée
GARDER_POIDS = False                                          # False : supprime ckpt_best.pth (≈ 0,5 Go) après chaque cible
PROCESSUS    = min(8, os.cpu_count() or 2)                    # chargement des images (n'affecte pas l'apprentissage)

## Installation

In [ ]:
import os, sys, subprocess, json, glob, hashlib, time, shutil
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(SORTIE, exist_ok=True)

if not os.path.isdir("/content/MM-WBHR"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCHE, DEPOT, "/content/MM-WBHR"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ekphrasis", "emoji"], check=True)

import torch
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > GPU"
print("GPU :", torch.cuda.get_device_name(0))

## Données
Images : archives des auteurs (lien de leur README), reprise du téléchargement et contrôle de taille.
Découpes : GitHub des auteurs (`jingjiang02/M2CAN`), empreintes SHA-256 vérifiées (identiques à nos runs Kaggle).
Les chemins sont ceux de `config.py` de M2CAN (`/workspaces/M2CAN1/...`).

In [ ]:
WS = "/workspaces/M2CAN1"
DRIVE_URL = "https://drive.usercontent.google.com/download?id={}&export=download&confirm=t"
ZIPS = {"AVA":  ("1DVJD5MwmnX0FecDYRq1FQuE39J0qC9Iz", 553246909, 4235),
        "PCCD": ("1BdXjEyvSiONG7h6jIli22lT9i6vy31I2", 886492747, 4235),
        "RPCD": ("1FWgVKJQuC6wcs_NqTBkPDujJZKlGSe79", 4161232968, 4235)}
GH = "https://raw.githubusercontent.com/jingjiang02/M2CAN/main/split"
SHA = {"AVA/train": "8add3a6ef56e39914a28f846127c447a21f762fe5e42a2262b2c48cdb30f2248",
       "AVA/val": "9aa9bdb0eb77d2509ab711a7ddc529aad1f53da77ee94b130bb7285bd9d47949",
       "PCCD/train": "77563b6c79dc45c59bc63bd6681e8bb75ae37b1aa9c472b2c8d3b3582dd50a9e",
       "PCCD/val": "ea25a79a49e4de8967bf314b1c6e3fde9fce008c2c0885c48bd6b49a9dbab7fb",
       "RPCD/train": "eb8a5f665a9891a9d80bbd557af83bc6844948861a738e6294a6faabfa1c46fa",
       "RPCD/val": "dc4c9351f7147029c1ae70142e8fc91da612e062e3aa3169325865e93a5a1882"}

def archive(nom):
    fid, taille, n_img = ZIPS[nom]
    fait = f"{WS}/datasets/.{nom}_ok"
    if os.path.exists(fait):
        print(f"{nom} : déjà prêt"); return
    os.makedirs("/content/zips", exist_ok=True)
    z = f"/content/zips/{nom}.zip"
    for essai in range(6):
        subprocess.run(["curl", "-L", "-sS", "--retry", "5", "--retry-delay", "10", "-C", "-", "-o", z, DRIVE_URL.format(fid)])
        recu = os.path.getsize(z) if os.path.exists(z) else 0
        if recu == taille:
            break
        if recu > taille or 0 < recu < 1_000_000:          # page HTML (quota Google Drive ?) : on repart de zéro
            os.remove(z)
        print(f"  {nom} : {recu}/{taille} octets, nouvel essai dans {30 * (essai + 1)} s"); time.sleep(30 * (essai + 1))
    else:
        raise RuntimeError(f"{nom}.zip incomplet : quota Google Drive des auteurs ? Réessayer plus tard.")
    os.makedirs(f"{WS}/datasets", exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", z, "-d", f"{WS}/datasets"], check=True)
    os.remove(z)
    open(fait, "w").close()
    print(f"{nom} : {taille / 1e9:.2f} Go téléchargés et décompressés")

for d in ("AVA", "PCCD", "RPCD"):                            # les 3 domaines servent de sources ou de cible
    archive(d)
    for s in ("train", "val"):
        p = f"{WS}/split/aesthetic/{d}/{s}.json"
        os.makedirs(os.path.dirname(p), exist_ok=True)
        if not os.path.exists(p):
            subprocess.run(["curl", "-L", "-sS", "--retry", "5", "-o", p, f"{GH}/aesthetic/{d}/{s}.json"], check=True)
        h = hashlib.sha256(open(p, "rb").read()).hexdigest()
        assert h == SHA[f"{d}/{s}"], f"découpe {d}/{s} différente de la référence"
print("découpes conformes")

## Entraînement
Une cible à la fois : configuration publiée de la cible, seuls changent le lot (8, valeur publiée ; le dépôt est réglé
à 4 pour Kaggle), le nombre de processus de chargement, la graine et le dossier de sortie. M2CAN évalue la cible à
chaque époque ; on rapporte le **dernier point** et, à titre indicatif, le **meilleur** (ce que rapporte l'article).

In [ ]:
import re, yaml
CODE = "/content/MM-WBHR/m2can"

def lire_exactitudes(log, n_test):
    """Exactitudes de la cible à chaque évaluation (lignes 'accuracy ... <n_test>' des rapports de classification)."""
    acc = []
    for l in open(log, encoding="utf-8", errors="replace"):
        m = re.search(r"accuracy\s+([0-9.]+)\s+(\d+)\s*$", l)
        if m and int(m.group(2)) == n_test:
            acc.append(float(m.group(1)))
    return acc

for cible in CIBLES:
    dossier = f"{SORTIE}/{cible}_s{GRAINE}"
    res_f = f"{dossier}/resultats.json"
    if os.path.exists(res_f) and json.load(open(res_f)).get("rc") == 0:
        print(f"{cible} : déjà fait", json.load(open(res_f))); continue
    os.makedirs(dossier, exist_ok=True)
    shutil.rmtree(f"{dossier}/run", ignore_errors=True)            # pas de reste d'un essai interrompu dans log.txt
    cfg = yaml.safe_load(open(f"{CODE}/msmm/scripts/config_{cible}.yaml"))
    cfg.setdefault("dataset", {}).update({"batch_size": LOT, "workers": PROCESSUS})
    cfg.setdefault("trainer", {})["epochs"] = EPOQUES
    cfg["seed"], cfg["snapshot_dir"] = GRAINE, f"{dossier}/run"   # absolu : M2CAN y écrit aussi ses poids (relatif = plantage)
    cp = f"{dossier}/config_{cible}.yaml"
    yaml.safe_dump(cfg, open(cp, "w"))
    print(f"=== {cible} : M2CAN de bout en bout, lot {LOT}, {EPOQUES} époques, graine {GRAINE}", flush=True)
    t0 = time.time()
    with open(f"{dossier}/journal.txt", "w") as j:
        rc = subprocess.run([sys.executable, "msmm/train.py", "--config", cp], cwd=CODE, stdout=j, stderr=subprocess.STDOUT).returncode
    n_test = len(json.load(open(f"{WS}/split/aesthetic/{cible}/val.json")))
    acc = lire_exactitudes(f"{dossier}/run/log.txt", n_test) if os.path.exists(f"{dossier}/run/log.txt") else []
    r = {"cible": cible, "graine": GRAINE, "rc": rc, "minutes": round((time.time() - t0) / 60, 1), "n_test": n_test,
         "exactitudes": acc, "dernier": acc[-1] if acc else None, "meilleur": max(acc) if acc else None}
    json.dump(r, open(res_f, "w"), indent=1)
    if not GARDER_POIDS:
        for f in glob.glob(f"{dossier}/run/*.pth"):
            os.remove(f)
    print(r if rc == 0 else f"ÉCHEC (code {rc}) : voir {dossier}/journal.txt", flush=True)

## Récapitulatif

In [ ]:
ARTICLE = {"AVA": 79.9, "PCCD": 69.8, "RPCD": 74.5}            # M2CAN, article (ResNet-50 + BERT, meilleure époque)
GELE = {"AVA": 69.62, "PCCD": 62.42, "RPCD": 59.94}            # M2CAN sur CLIP gelé, extracteur 5e-5 (nos runs Kaggle)
lignes = []
for cible in CIBLES:
    f = f"{SORTIE}/{cible}_s{GRAINE}/resultats.json"
    if os.path.exists(f):
        r = json.load(open(f))
        lignes.append((cible, r["dernier"], r["meilleur"], ARTICLE[cible], GELE[cible], r["minutes"], r["rc"], len(r["exactitudes"])))
print(f"{'cible':6s} {'dernier':>8s} {'meilleur':>9s} {'article':>8s} {'gelé':>6s} {'minutes':>8s} {'rc':>3s} {'évals':>6s}")
for c, d, m, a, g, mn, rc, n in lignes:
    print(f"{c:6s} {d * 100 if d is not None else float('nan'):8.2f} {m * 100 if m is not None else float('nan'):9.2f} {a:8.1f} {g:6.2f} {mn:8.1f} {rc:3d} {n:6d}")
json.dump([dict(zip(["cible", "dernier", "meilleur", "article", "gele", "minutes", "rc", "evaluations"], l)) for l in lignes],
          open(f"{SORTIE}/recapitulatif.json", "w"), indent=1)
print(f"\nRécapitulatif écrit dans {SORTIE}/recapitulatif.json")